# Data transcript analysis

The show_consented toggle can ve check to limit the results to just those who have consented.

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/main/Data_transcript_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Google API

In [ ]:
# @title Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FILE
from google.colab import userdata

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')
# PACKAGE_PATH = userdata.get('PACKAGE_PATH')


# @title ## Install pyDrive  {"form-width":"20%"}

# @markdown ---
# @markdown Installing PyDrive
# @markdown
# @markdown Thie is necessary if using the Google API to call files by their file_id

# !pip install PyDrive

import sys
# 2. Tell Python to look in your custom folder
# sys.path.append(PACKAGE_PATH)
sys.path.append('src')
# 3. Import your file!
from GoogleFunctions import GoogleDocumentManager, extract_file_id
from local_io import read_csv_from_id


## Read in data

data read in will be merged using the `user` column

- userprorile data (`username` renamed to `user` if necessary)
- consent file
- transcript data

In [ ]:

# @title Read in userprofile file {"form-width":"20%"}
userprofile_file_link = "https://drive.google.com/file/d/1_iV_oqPDvZQ1BXbabl8u0pjmn0SB65g7/view?usp=drive_link" # @param {"type":"string"}
userprofile_file_link_id = extract_file_id(userprofile_file_link)
show_userprofile = False # @param {"type":"boolean"}


userprofile_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, userprofile_file_link_id, show_userprofile)

In [ ]:

# @title Read in consent file {"form-width":"20%"}
consent_file_link = "https://drive.google.com/file/d/1RUQV3zuCEn2mmzHCj3ilddUGo3AHhjUr/view?usp=drive_link" # @param {"type":"string"}
consent_file_link_id = extract_file_id(consent_file_link)
show_consent = False # @param {"type":"boolean"}

consent_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, consent_file_link_id, show_consent)

In [ ]:

# @title Read in transcript file {"form-width":"20%"}
trans_file_link = "https://drive.google.com/file/d/18rb0--aU4BhzLSOXkq-w0yziiagFwXri/view?usp=drive_link" # @param {"type":"string"}
trans_file_link_id = extract_file_id(trans_file_link)
show_trans = False # @param {"type":"boolean"}
show_remove_staff_data = False # @param {"type":"boolean"}

trans_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, trans_file_link_id, show_trans)

if show_remove_staff_data:
    from data_scrub import remove_staff
    userprofile_df, trans_df = remove_staff(userprofile_df,trans_df)

## Process data

In [ ]:
import pandas as pd
import re
import ast
import json

####
# Merge consent into transcript
###

combined_df = pd.merge(consent_df, trans_df, on='user', how='inner')

print("Consent Data Combined into DataFrame.")
# display(combined_df.head())

####
# Split up question column
###

# Define the regex pattern to capture the components
pattern = r"(.+) - Assignment (\d+) Question (\d+) - (.+)"

# Apply the regex to the 'question' column and create new columns
combined_df[['Course', 'lab_number', 'question_number', 'question_text']] = \
    combined_df['question'].str.extract(pattern)

# Convert lab_number and question_number to numeric types
combined_df['lab_number'] = pd.to_numeric(combined_df['lab_number'])
combined_df['question_number'] = pd.to_numeric(combined_df['question_number'])

# Format lab_number and question_number to add leading zeros if single digit
combined_df['lab_number'] = combined_df['lab_number'].astype(int).astype(str).str.zfill(2)
combined_df['question_number'] = combined_df['question_number'].astype(int).astype(str).str.zfill(2)

# Replace the 'question' column with the new formatted string
combined_df['question'] = 'Lab ' + combined_df['lab_number'] + '-Q' + combined_df['question_number']

print(' Question Data parsed and updated.')

####
# Convert the JSON format to lists of dicts
###

# Function to parse message strings into list of dictionaries
def parse_messages(message_string):
    if pd.isna(message_string) or message_string == '[]':
        return []
    try:
        return json.loads(message_string)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(message_string)
        except (ValueError, SyntaxError):
            return [] # Return empty list if parsing fails

# Apply the parsing function to the 'all_messages' column
combined_df['all_messages'] = combined_df['all_messages'].apply(parse_messages)

# Count message objects in 'all_messages' and add to 'interactions' column
combined_df['interactions'] = combined_df['all_messages'].apply(len)

import json
combined_df = combined_df[combined_df['all_messages'].apply(lambda x: len(x) > 0)]

print("Combined DataFrame with split question columns and interactions:")
# display(combined_df.head())


## Filter functions

In [ ]:
# @title convert_utc_to_la function
from datetime import datetime
import pytz

def convert_utc_to_la(iso_str):
    # 1. Parse the string
    dt = datetime.fromisoformat(iso_str)

    # 2. Check if it is UTC (+00:00)
    # This ensures we only convert the ones that need it
    if dt.utcoffset().total_seconds() == 0:
        la_tz = pytz.timezone('America/Los_Angeles')
        # .astimezone() performs the clock-shift math
        return dt.astimezone(la_tz)

    # If it's already in another timezone, return as is
    return dt

# Example: 8:00 PM UTC (20:00)
# In LA (Winter), this should be 12:00 PM (noon)
print(convert_utc_to_la("2026-01-28T20:00:00+00:00"))
# Output: 2026-01-28 12:00:00-08:00

In [ ]:
# @title ipywidgets functions
from datetime import datetime
import convert_utc_to_la

# This function will be called when the course dropdown changes
def on_course_change(change):
    with output_widget:
        clear_output(wait=True)
        selected_course = change.new

        if selected_course == "Select Course":
            lab_dropdown.options = ['Select Lab']
            lab_dropdown.value = 'Select Lab'
            lab_dropdown.disabled = True

            # print("Please select a Course from the dropdown.")
        else:
            # Filter combined_df based on selected_course
            course_filtered_df = combined_df[combined_df['Course'] == selected_course]

            # Update lab_dropdown options based on the course_filtered_df
            unique_labs = course_filtered_df['lab_number'].dropna().unique().tolist()
            if len(unique_labs) > 0:
                lab_dropdown.options = ['Select Lab'] + sorted(unique_labs)
            else:
                lab_dropdown.options = ['Select Lab']

            lab_dropdown.value = 'Select Lab'
            lab_dropdown.disabled = False

        # Reset question and user dropdowns (on_lab_change will populate them)
        question_dropdown.options = ['Select Question']
        question_dropdown.value = 'Select Question'
        question_dropdown.disabled = True
        user_dropdown.options = ['Select User']
        user_dropdown.value = 'Select User'
        user_dropdown.disabled = True

# This function will be called when the lab dropdown changes
def on_lab_change(change):
    with output_widget:
        clear_output(wait=True)
        selected_course = course_dropdown.value
        selected_lab = change.new

        # Apply course filter
        if selected_course != "Select Course":
            df_for_next_filter = combined_df[combined_df['Course'] == selected_course]

            # Apply lab filter
            if selected_lab != "Select Lab":
                df_for_next_filter = df_for_next_filter[df_for_next_filter['lab_number'] == selected_lab]

                # Update question_dropdown options
                unique_questions = df_for_next_filter['question_number'].dropna().unique().tolist()

                if len(unique_questions) > 0:
                    question_dropdown.options = ['Select Question'] + sorted(unique_questions)
                else:
                    question_dropdown.options = ['Select Question']
                question_dropdown.disabled = False
            else:
                question_dropdown.options = ['Select Question']
                question_dropdown.disabled = True

            question_dropdown.value = 'Select Question'

            # Reset user dropdown (on_question_change will populate it)
            user_dropdown.options = ['Select User']
            user_dropdown.value = 'Select User'
            user_dropdown.disabled = True

# This function will be called when the question dropdown changes
def on_question_change(change):
    with output_widget:
        clear_output(wait=True)
        selected_course = course_dropdown.value
        selected_lab = lab_dropdown.value
        selected_question = change.new

        # Apply course filter
        if selected_course != "Select Course":
            df_for_next_filter = combined_df[combined_df['Course'] == selected_course]

            # Apply lab filter
            if selected_lab != "Select Lab":
                df_for_next_filter = df_for_next_filter[df_for_next_filter['lab_number'] == selected_lab]

                # Apply question filter
                if selected_question != "Select Question":
                    df_for_next_filter = df_for_next_filter[df_for_next_filter['question_number'] == selected_question]

                    # Update user_dropdown options
                    unique_users = df_for_next_filter['user'].dropna().unique().tolist()
                    if len(unique_users) > 0:
                        user_dropdown.options = ['Select User'] + sorted(unique_users)
                        user_dropdown.disabled = False
                    else:
                        user_dropdown.options = ['Select User']
                        user_dropdown.disabled = False
                else:
                    user_dropdown.options = ['Select User']
                    user_dropdown.disabled = True
                user_dropdown.value = 'Select User'


# This function will be called when the user dropdown changes
def on_user_change(change):
    with output_widget:
        clear_output(wait=True)
        selected_course = course_dropdown.value
        selected_lab = lab_dropdown.value
        selected_question = question_dropdown.value
        selected_user = change.new

        # Apply course filter
        if selected_course != "Select Course":
            df_to_display = combined_df[combined_df['Course'] == selected_course]

            # Apply lab filter
            if selected_lab != "Select Lab":
                df_to_display = df_to_display[df_to_display['lab_number'] == selected_lab]

                # Apply question filter
                if selected_question != "Select Question":
                    df_to_display = df_to_display[df_to_display['question_number'] == selected_question]

                    # Apply user filter
                    if selected_user != "Select User":
                        df_to_display = df_to_display[df_to_display['user'] == selected_user]

                        # Final display based on all filters
                        course_str = f"Course: {selected_course}"
                        lab_str = f"Lab: {selected_lab}"
                        question_str = f"Question: {selected_question}"
                        user_str = f"User: {selected_user}"

                        print(f"Displaying data for {course_str}, {lab_str}, {question_str}, {user_str}.")
                        if df_to_display.empty:
                            print("No data matches the selected filters.")
                        else:
                            df_to_display = explode_json_messages(df_to_display)
                            # display(df_to_display)

                            for row in df_to_display.itertuples(index=False):
                              row_dict = row._asdict()
                              # print(row_dict.keys())

                              # fix timestamps to Pacific Time zone
                              fixed_timestamp = convert_utc_to_la(row.timestamp)
                              dt_obj = datetime.fromisoformat(str(fixed_timestamp))
                              natural_format = dt_obj.strftime("%b %d, %Y, %I:%M:%S %p")
                              print(f"\n{natural_format}\n {row.sender}: {row.text}\n")

                              # print transcript
                              if row.sender == 'Bot':
                                if 'rating' in row_dict.keys() and str(row.rating) != 'nan':
                                  print(f"Rating: {row.rating}")
                                if 'data' in row_dict.keys() and row.data == True:
                                  print(f"Spreadsheet data used: {row.data}")
                                print("------------------------\n")
                    else:
                        print("Select a User")
                else:
                    print("Select a Question")
            else:
                print("Select a Lab")
        else:
            print("Select a Course")


# Output Display

In [ ]:
# @title Dropdown Filters {"form-width":"20%"}
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Course Dropdown ---
# Get unique course values
unique_courses_all = ['Select Course', 'All'] + sorted(combined_df['Course'].dropna().unique().tolist())
course_dropdown = widgets.Dropdown(
    options=unique_courses_all,
    value='Select Course', # Initial value
    description='Select Course:',
    disabled=False,
)

# --- Lab Dropdown ---
# Initialize with default options; on_course_change will populate it
lab_dropdown = widgets.Dropdown(
    options=['Select Lab', 'All'], # Initial options
    value='Select Lab', # Initial value
    description='Select Lab:',
    disabled=True,
)

# --- Question Dropdown ---
# Initialize with default options; on_lab_change will populate it
question_dropdown = widgets.Dropdown(
    options=['Select Question', 'All'], # Initial options
    value='Select Question', # Initial value
    description='Select Question:',
    disabled=True,
)

# --- User Dropdown ---
# Initialize with default options; on_question_change will populate it
user_dropdown = widgets.Dropdown(
    options=['Select User', 'All'], # Initial options
    value='Select User', # Initial value
    description='Select User:',
    disabled=True,
)

# --- Output Widget ---
# --- Observers ---
# Link on_course_change to the course dropdown
course_dropdown.observe(on_course_change, names='value')
# Link on_lab_change to the lab dropdown
lab_dropdown.observe(on_lab_change, names='value')
# Link on_question_change to the question dropdown
question_dropdown.observe(on_question_change, names='value')
# Link on_user_change to the user dropdown
user_dropdown.observe(on_user_change, names='value')

# --- Display Widgets ---
display(course_dropdown, lab_dropdown, question_dropdown, user_dropdown)

# --- Initial State ---
# Set initial course to 'All' to populate all dropdowns and display all data
# This will trigger the chain of observers: on_course_change -> on_lab_change -> on_question_change -> on_user_change
# Ensure this happens AFTER the widgets are displayed and observers are set up.
course_dropdown.value = 'Select Course'
output_widget = widgets.Output()


In [ ]:
# @title Chat Transcript
display(output_widget)